# M5 Forecasting — RNN Model Comparison Using Final Feature Set

This notebook trains a lightweight RNN-family model for model exploration and comparison with the main LightGBM model.

The setup is aligned with the team's `07a_m5_model_exploration.ipynb`:

- Input file: `long_df_with_features.parquet`
- Active-row filter: keep `is_active == 1`
- Drop rows with missing required lag/rolling/price features
- Feature set: same final engineered feature set used by the model exploration notebook
- Validation window: final 28 days
- Competition-relevant metric: RMSE / MAE plus a simplified item-store WRMSSE proxy
- Main difference: RNN converts the long-format data into sequences. Each sample uses the previous 28 days of features to predict next-day demand.

Performance note: sequence construction is vectorized with NumPy to avoid slow nested Python loops.


## 1. Setup

Run this notebook in Google Colab. Update `FEATURE_DATA_PATH` if your Drive folder path is different.


In [1]:
# If running in Google Colab, uncomment and run:
# from google.colab import drive
# drive.mount('/content/drive')
# Then update FEATURE_DATA_PATH below to your Drive location.


Mounted at /content/drive


In [2]:
import os
import gc
import time
import random
import numpy as np
import pandas as pd

from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, SimpleRNN, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)


TensorFlow version: 2.20.0


## 2. Configuration

This notebook does **not** sample `id` series, because we want the prediction window to remain aligned with the team's other model exploration notebook.

To keep RNN training practical, we only cap the **number of training sequences**. The validation window is still the final 28 days.


In [3]:
# =========================
# File path
# =========================
# Option 1: Google Drive path. Update this if your folder name is different.
# Update FEATURE_DATA_PATH to match your local environment.
FEATURE_DATA_PATH = "../results/long_df_with_features.parquet"  # output of notebook 03

# Option 2: local uploaded file fallback, useful if you upload the parquet directly to Colab.
LOCAL_FALLBACK_PATHS = [
    "/content/long_df_with_features.parquet",
    "./long_df_with_features.parquet",
    "./data/long_df_with_features.parquet",
]

if not os.path.exists(FEATURE_DATA_PATH):
    for p in LOCAL_FALLBACK_PATHS:
        if os.path.exists(p):
            FEATURE_DATA_PATH = p
            break

print("Using file:", FEATURE_DATA_PATH)
print("File exists:", os.path.exists(FEATURE_DATA_PATH))

# =========================
# Experiment settings
# =========================
VALID_DAYS = 28          # same validation horizon as teammate notebook
SEQ_LEN = 28             # use past 28 days of features to predict next-day demand

# Training all possible RNN windows can be heavy. This is similar in spirit to
# the teammate notebook sampling 500k training rows for RF/Ridge.
MAX_TRAIN_SEQUENCES = 500_000

# Keep all validation sequences by default. If Colab runs out of RAM, set this to
# a number like 200_000, but report that validation was sampled.
MAX_VALID_SEQUENCES = None

RNN_TYPE = "GRU"   # options: "SimpleRNN" or "GRU"
RNN_UNITS = 32
DROPOUT_RATE = 0.10
EPOCHS = 15
BATCH_SIZE = 512


Using file: /content/drive/MyDrive/long_df_with_features.parquet
File exists: True


## 3. Load final feature dataset

This file already contains the final selected engineered features. We do **not** re-run feature engineering here.


In [4]:
needed_cols = [
    "id", "item_id", "dept_id", "cat_id", "store_id", "state_id",
    "date", "demand", "is_active",
    "sell_price", "price_change", "is_promo",
    "dow", "month",
    "lag_7", "lag_28", "rmean_7", "rmean_28",
    "lag_56", "snap_flag", "is_event"
]

df = pd.read_parquet(FEATURE_DATA_PATH, columns=needed_cols)
df["date"] = pd.to_datetime(df["date"])

print(f"Loaded shape: {df.shape}")
print(f"Date range: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Unique series: {df['id'].nunique():,}")

print("\nColumn types and missing values:")
for col in df.columns:
    print(f"  {col:25s} {str(df[col].dtype):12s} NaN={df[col].isna().sum():>10,}")


Loaded shape: (16098720, 21)
Date range: 2014-12-12 → 2016-05-22
Unique series: 30,490

Column types and missing values:
  id                        object       NaN=         0
  item_id                   object       NaN=         0
  dept_id                   object       NaN=         0
  cat_id                    object       NaN=         0
  store_id                  object       NaN=         0
  state_id                  object       NaN=         0
  date                      datetime64[ns] NaN=         0
  demand                    int16        NaN=         0
  is_active                 int8         NaN=         0
  sell_price                float32      NaN=         0
  price_change              float32      NaN=         0
  is_promo                  int8         NaN=         0
  dow                       int8         NaN=         0
  month                     int8         NaN=         0
  lag_7                     float32      NaN=   213,430
  lag_28                    float32  

## 4. Match teammate preprocessing

This section is intentionally aligned with `07a_m5_model_exploration.ipynb`:

1. Keep only active rows: `is_active == 1`
2. Drop rows where required lag/rolling/price features are missing
3. Use the same final feature list


In [5]:
# =========================
# Filter to active rows & drop NaN
# =========================
df = df[df["is_active"] == 1].reset_index(drop=True)
print(f"After active filter: {df.shape}")

# Required features — same as teammate notebook
required_cols = ["lag_7", "lag_28", "rmean_7", "rmean_28", "lag_56", "sell_price"]
df = df.dropna(subset=required_cols).reset_index(drop=True)
print(f"After dropping NaN in required features: {df.shape}")


After active filter: (15974786, 21)
After dropping NaN in required features: (14315186, 21)


In [6]:
# =========================
# Define feature columns
# =========================
categorical_cols = ["item_id", "dept_id", "cat_id", "store_id", "state_id"]

numerical_cols = [
    "sell_price", "price_change", "is_promo",
    "dow", "month",
    "lag_7", "lag_28", "lag_56",
    "rmean_7", "rmean_28",
    "snap_flag", "is_event",
]

target_col = "demand"
feature_cols = categorical_cols + numerical_cols

# Keep only columns that exist, as a safety check
feature_cols = [c for c in feature_cols if c in df.columns]
categorical_cols = [c for c in categorical_cols if c in feature_cols]
numerical_cols = [c for c in numerical_cols if c in feature_cols]

print(f"Categorical features: {len(categorical_cols)}")
print(f"Numerical features: {len(numerical_cols)}")
print(f"Total features: {len(feature_cols)}")
print(feature_cols)


Categorical features: 5
Numerical features: 12
Total features: 17
['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'sell_price', 'price_change', 'is_promo', 'dow', 'month', 'lag_7', 'lag_28', 'lag_56', 'rmean_7', 'rmean_28', 'snap_flag', 'is_event']


## 5. Same final 28-day validation window

This matches the teammate notebook's validation split exactly:

```python
valid_start = max_date - pd.Timedelta(days=VALID_DAYS - 1)
train_df = df[df["date"] < valid_start]
valid_df = df[df["date"] >= valid_start]
```


In [7]:
max_date = df["date"].max()
valid_start = max_date - pd.Timedelta(days=VALID_DAYS - 1)

train_rows = df["date"] < valid_start
valid_rows = df["date"] >= valid_start

print(f"Validation window: {valid_start.date()} → {max_date.date()}")
print(f"Train rows: {train_rows.sum():,}")
print(f"Validation rows: {valid_rows.sum():,}")


Validation window: 2016-04-25 → 2016-05-22
Train rows: 13,461,466
Validation rows: 853,720


## 6. Encode categorical features and scale numeric features

RNNs require numeric inputs. We fit encoders/scalers using the training period only to avoid validation leakage.


In [8]:
# Sort before preprocessing and sequence construction
df = df.sort_values(["id", "date"]).reset_index(drop=True)

model_df = df[["id", "date", target_col] + feature_cols].copy()

# Prepare categoricals
for c in categorical_cols:
    model_df[c] = model_df[c].astype("string").fillna("missing")

# Prepare numerics
for c in numerical_cols:
    model_df[c] = pd.to_numeric(model_df[c], errors="coerce")

# Fit preprocessing on training period only
preprocess_train_rows = model_df["date"] < valid_start

encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
if categorical_cols:
    encoder.fit(model_df.loc[preprocess_train_rows, categorical_cols])
    model_df.loc[:, categorical_cols] = encoder.transform(model_df.loc[:, categorical_cols]).astype(np.float32)

scaler = StandardScaler()
if numerical_cols:
    # Required lag columns were already dropped if missing; remaining missing values are safety-filled.
    model_df.loc[:, numerical_cols] = model_df.loc[:, numerical_cols].fillna(0)
    scaler.fit(model_df.loc[preprocess_train_rows, numerical_cols])
    model_df.loc[:, numerical_cols] = scaler.transform(model_df.loc[:, numerical_cols]).astype(np.float32)

# Final safety conversion
model_df[feature_cols] = model_df[feature_cols].astype(np.float32).fillna(0)

print(model_df[feature_cols].head())


/tmp/ipykernel_2091/707062477.py:20: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[   0.    0.    0. ... 3048. 3048. 3048.]' has dtype incompatible with string, please explicitly cast to a compatible dtype first.
  model_df.loc[:, categorical_cols] = encoder.transform(model_df.loc[:, categorical_cols]).astype(np.float32)
/tmp/ipykernel_2091/707062477.py:20: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0. 0. 0. ... 6. 6. 6.]' has dtype incompatible with string, please explicitly cast to a compatible dtype first.
  model_df.loc[:, categorical_cols] = encoder.transform(model_df.loc[:, categorical_cols]).astype(np.float32)
/tmp/ipykernel_2091/707062477.py:20: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0. 0. 0. ... 2. 2. 2.]' has dtype incompatible with string, please exp

   item_id  dept_id  cat_id  store_id  state_id  sell_price  price_change  \
0      0.0      0.0     0.0       0.0       0.0   -0.633264     -0.001702   
1      0.0      0.0     0.0       0.0       0.0   -0.633264     -0.001702   
2      0.0      0.0     0.0       0.0       0.0   -0.633264     -0.001702   
3      0.0      0.0     0.0       0.0       0.0   -0.633264     -0.001702   
4      0.0      0.0     0.0       0.0       0.0   -0.633264     -0.001702   

   is_promo       dow     month     lag_7    lag_28    lag_56   rmean_7  \
0 -0.023479  0.492987 -1.140863 -0.357278 -0.354902 -0.070361 -0.230345   
1 -0.023479  0.992722 -1.140863  0.478121  0.204019  0.210891 -0.230345   
2 -0.023479  1.492456 -1.140863 -0.357278  1.042399  0.210891 -0.230345   
3 -0.023479 -1.505951 -1.140863 -0.357278 -0.075442 -0.070361 -0.230345   
4 -0.023479 -1.006216 -1.140863 -0.357278 -0.354902 -0.070361 -0.230345   

   rmean_28  snap_flag  is_event  
0 -0.076892   1.427226 -0.301338  
1 -0.076892   1.

## 7. Create RNN sequences with vectorized NumPy logic

Each RNN sample uses the previous `SEQ_LEN` rows for the same `id` to predict the current day's `demand`.

- Training target dates: before validation start
- Validation target dates: inside the final 28-day window
- Training sequences are capped at `MAX_TRAIN_SEQUENCES` for runtime, similar to the teammate notebook's training sample.
- Validation sequences use the final 28-day window. If memory is an issue, set `MAX_VALID_SEQUENCES`.

This version avoids the slow nested Python loop over every series and every date. It first creates vectorized `(start_pos, target_pos)` arrays, then builds 3D RNN arrays with NumPy indexing.


In [9]:
def collect_sequence_indices_vectorized(data, seq_len, valid_start):
    """Collect RNN sequence start/target positions using vectorized group ranges.

    Because `data` is sorted by id/date and reset_index was called, each id group is
    contiguous. For each target position t, the input window is [t-seq_len, ..., t-1].
    """
    # groupby(sort=False) keeps current id order after sorting by id/date
    sizes = data.groupby("id", sort=False).size().to_numpy()
    group_starts = np.r_[0, np.cumsum(sizes)[:-1]]

    target_chunks = []
    for start, n in zip(group_starts, sizes):
        if n > seq_len:
            target_chunks.append(np.arange(start + seq_len, start + n, dtype=np.int64))

    if not target_chunks:
        empty = np.array([], dtype=np.int64)
        return empty, empty, empty, empty

    target_pos = np.concatenate(target_chunks)
    start_pos = target_pos - seq_len

    date_values = data["date"].to_numpy(dtype="datetime64[ns]")
    valid_start64 = np.datetime64(valid_start)

    is_valid = date_values[target_pos] >= valid_start64
    train_start = start_pos[~is_valid]
    train_target = target_pos[~is_valid]
    valid_start_pos = start_pos[is_valid]
    valid_target = target_pos[is_valid]

    return train_start, train_target, valid_start_pos, valid_target


def sample_positions(start_pos, target_pos, max_n, seed=42):
    """Randomly sample sequence positions while preserving start/target alignment."""
    n = len(target_pos)
    if max_n is None or n <= max_n:
        return start_pos, target_pos

    rng = np.random.default_rng(seed)
    sampled_idx = rng.choice(n, size=max_n, replace=False)
    sampled_idx.sort()
    return start_pos[sampled_idx], target_pos[sampled_idx]


def build_arrays_from_positions(data, start_pos, target_pos, feature_cols, target_col, seq_len):
    """Build X/y arrays with vectorized NumPy indexing instead of a Python loop."""
    n_samples = len(target_pos)
    n_features = len(feature_cols)

    if n_samples == 0:
        X = np.empty((0, seq_len, n_features), dtype=np.float32)
        y = np.empty((0,), dtype=np.float32)
        target_dates = np.empty((0,), dtype="datetime64[ns]")
        target_ids = np.empty((0,), dtype=object)
        return X, y, target_dates, target_ids

    feature_values = data[feature_cols].to_numpy(dtype=np.float32, copy=False)
    target_values = data[target_col].to_numpy(dtype=np.float32, copy=False)
    date_values = data["date"].to_numpy(dtype="datetime64[ns]")
    id_values = data["id"].to_numpy()

    # Shape: (n_samples, seq_len). Each row contains the previous seq_len positions.
    window_offsets = np.arange(seq_len, dtype=np.int64)
    window_indices = start_pos[:, None] + window_offsets[None, :]

    X = feature_values[window_indices]
    y = target_values[target_pos]
    target_dates = date_values[target_pos]
    target_ids = id_values[target_pos]

    return X, y, target_dates, target_ids


t0 = time.time()
train_start_pos, train_target_pos, valid_start_pos, valid_target_pos = collect_sequence_indices_vectorized(
    model_df, SEQ_LEN, valid_start
)

print(f"All candidate training sequences: {len(train_target_pos):,}")
print(f"All candidate validation sequences: {len(valid_target_pos):,}")

train_start_pos, train_target_pos = sample_positions(
    train_start_pos, train_target_pos, MAX_TRAIN_SEQUENCES, seed=SEED
)
valid_start_pos, valid_target_pos = sample_positions(
    valid_start_pos, valid_target_pos, MAX_VALID_SEQUENCES, seed=SEED
)

print(f"Used training sequences: {len(train_target_pos):,}")
print(f"Used validation sequences: {len(valid_target_pos):,}")

X_train, y_train, train_dates, train_ids = build_arrays_from_positions(
    model_df, train_start_pos, train_target_pos, feature_cols, target_col, SEQ_LEN
)

X_valid, y_valid, valid_dates, valid_ids = build_arrays_from_positions(
    model_df, valid_start_pos, valid_target_pos, feature_cols, target_col, SEQ_LEN
)

print(f"Sequence construction time: {time.time() - t0:.2f} seconds")
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_valid shape:", X_valid.shape)
print("y_valid shape:", y_valid.shape)
print("Validation target date range:", pd.to_datetime(valid_dates).min(), "→", pd.to_datetime(valid_dates).max())


All candidate training sequences: 12,607,746
All candidate validation sequences: 853,720
Used training sequences: 500,000
Used validation sequences: 853,720
Sequence construction time: 6.06 seconds
X_train shape: (500000, 28, 17)
y_train shape: (500000,)
X_valid shape: (853720, 28, 17)
y_valid shape: (853720,)
Validation target date range: 2016-04-25 00:00:00 → 2016-05-22 00:00:00


## 8. Build a lightweight RNN model

This model is intentionally simple because the goal is comparison, not heavy neural-network tuning.


In [10]:
n_features = X_train.shape[2]

if RNN_TYPE == "GRU":
    rnn_layer = GRU(RNN_UNITS)
elif RNN_TYPE == "SimpleRNN":
    rnn_layer = SimpleRNN(RNN_UNITS)
else:
    raise ValueError("RNN_TYPE must be either 'SimpleRNN' or 'GRU'")

model = Sequential([
    Input(shape=(SEQ_LEN, n_features)),
    rnn_layer,
    Dropout(DROPOUT_RATE),
    Dense(32, activation="relu"),
    Dense(1)
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=[tf.keras.metrics.MeanAbsoluteError(name="mae")]
)

model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 32)             │         4,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,985 (23.38 KB)

 Trainable params: 5,985 (23.38 KB)

 Non-trainable params: 0 (0.00 B)

## 9. Train model

In [11]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

train_t0 = time.time()
history = model.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop],
    verbose=1
)
train_time = time.time() - train_t0
print(f"Training time: {train_time:.2f} seconds")


Epoch 1/15
977/977 ━━━━━━━━━━━━━━━━━━━━ 63s 62ms/step - loss: 8.1597 - mae: 1.2605 - val_loss: 5.7941 - val_mae: 1.1237
Epoch 2/15
977/977 ━━━━━━━━━━━━━━━━━━━━ 58s 59ms/step - loss: 6.4967 - mae: 1.1768 - val_loss: 5.5692 - val_mae: 1.1698
Epoch 3/15
977/977 ━━━━━━━━━━━━━━━━━━━━ 58s 59ms/step - loss: 6.1971 - mae: 1.1536 - val_loss: 5.8077 - val_mae: 1.1982
Epoch 4/15
977/977 ━━━━━━━━━━━━━━━━━━━━ 57s 58ms/step - loss: 6.0889 - mae: 1.1412 - val_loss: 5.2540 - val_mae: 1.1329
Epoch 5/15
977/977 ━━━━━━━━━━━━━━━━━━━━ 57s 59ms/step - loss: 5.9839 - mae: 1.1359 - val_loss: 5.1004 - val_mae: 1.1492
Epoch 6/15
977/977 ━━━━━━━━━━━━━━━━━━━━ 56s 57ms/step - loss: 5.9464 - mae: 1.1341 - val_loss: 5.2195 - val_mae: 1.1499
Epoch 7/15
977/977 ━━━━━━━━━━━━━━━━━━━━ 57s 59ms/step - loss: 6.1929 - mae: 1.1576 - val_loss: 5.2910 - val_mae: 1.1472
Epoch 8/15
977/977 ━━━━━━━━━━━━━━━━━━━━ 58s 60ms/step - loss: 6.1233 - mae: 1.1668 - val_loss: 5.5175 - val_mae: 1.1658
Training time: 465.45 seconds


## 10. Evaluate validation performance

We report RMSE and MAE first, then compute the same simplified item-store WRMSSE proxy used by the teammate notebook.


In [12]:
valid_pred = model.predict(X_valid, batch_size=BATCH_SIZE).reshape(-1)

# Demand cannot be negative
valid_pred = np.clip(valid_pred, 0, None)

rmse = mean_squared_error(y_valid, valid_pred) ** 0.5
mae = mean_absolute_error(y_valid, valid_pred)

print(f"RMSE: {rmse:.6f}")
print(f"MAE:  {mae:.6f}")
print(f"Mean actual:     {np.mean(y_valid):.6f}")
print(f"Mean prediction: {np.mean(valid_pred):.6f}")


1668/1668 ━━━━━━━━━━━━━━━━━━━━ 23s 14ms/step
RMSE: 2.258386
MAE:  1.149222
Mean actual:     1.442820
Mean prediction: 1.425817


## 11. WRMSSE proxy

This matches the teammate notebook's simplified item-store level WRMSSE proxy. It is not the full official M5 hierarchical evaluator, but it makes the RNN comparison more consistent with the Random Forest and Ridge exploration.


In [13]:
# =========================
# WRMSSE item-store proxy
# =========================
# Important:
# Use the original df here, not model_df, because model_df has scaled sell_price.
# WRMSSE weights should be based on original dollar sales = demand * sell_price.

train_for_scale = df.loc[
    df["date"] < valid_start,
    ["id", "date", "demand", "sell_price"]
].copy()

valid_for_wrmsse = pd.DataFrame({
    "id": valid_ids,
    "date": pd.to_datetime(valid_dates),
    "demand": y_valid,
})

def compute_wrmsse_proxy(train_data, valid_data, preds):
    """Simplified WRMSSE at the item-store level, aligned with teammate notebook."""

    # 1. Scale: mean squared diff of training demand per series
    scales = []
    for sid, g in train_data.groupby("id", sort=False):
        y = g.sort_values("date")["demand"].values
        nz = np.flatnonzero(y)

        if len(nz) <= 1:
            scales.append((sid, np.nan))
        else:
            y_trim = y[nz[0]:]
            diff = np.diff(y_trim)
            scale = np.mean(diff ** 2) if len(diff) > 0 else np.nan
            scales.append((sid, scale))

    scale_df = pd.DataFrame(scales, columns=["id", "scale"])
    scale_df = scale_df[scale_df["scale"] > 0].copy()

    # 2. Weights: dollar sales in last 28 training days
    last_date = train_data["date"].max()
    w_start = last_date - pd.Timedelta(days=27)

    wt = (
        train_data[train_data["date"] >= w_start]
        .assign(dollars=lambda x: x["demand"] * x["sell_price"])
        .groupby("id", as_index=False)["dollars"].sum()
    )

    total = wt["dollars"].sum()
    if total > 0:
        wt["weight"] = wt["dollars"] / total
    else:
        wt["weight"] = 1 / len(wt)

    # 3. Series-level RMSE
    eval_df = valid_data[["id"]].copy()
    eval_df["actual"] = valid_data["demand"].values
    eval_df["pred"] = preds

    series_rmse = (
        eval_df.groupby("id")
        .apply(lambda g: np.sqrt(np.mean((g["actual"].values - g["pred"].values) ** 2)))
        .reset_index(name="rmse")
    )

    # 4. Combine
    merged = (
        series_rmse
        .merge(scale_df, on="id", how="inner")
        .merge(wt[["id", "weight"]], on="id", how="inner")
    )

    merged["rmsse"] = merged["rmse"] / np.sqrt(merged["scale"])
    merged = merged.replace([np.inf, -np.inf], np.nan).dropna(subset=["rmsse", "weight"])

    merged["weight"] = merged["weight"] / merged["weight"].sum()

    return (merged["weight"] * merged["rmsse"]).sum()

rnn_wrmsse = compute_wrmsse_proxy(train_for_scale, valid_for_wrmsse, valid_pred)
print(f"WRMSSE Proxy: {rnn_wrmsse:.6f}")

WRMSSE Proxy: 0.863086


/tmp/ipykernel_2091/3975959074.py:62: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: np.sqrt(np.mean((g["actual"].values - g["pred"].values) ** 2)))


## 12. Save RNN results and predictions

In [14]:
results = pd.DataFrame([
    {
        "model": RNN_TYPE,
        "seq_len": SEQ_LEN,
        "validation_window_start": valid_start.date(),
        "validation_window_end": max_date.date(),
        "n_series_total": model_df["id"].nunique(),
        "n_features": len(feature_cols),
        "train_sequences": len(y_train),
        "valid_sequences": len(y_valid),
        "max_train_sequences": MAX_TRAIN_SEQUENCES,
        "max_valid_sequences": MAX_VALID_SEQUENCES,
        "rmse": rmse,
        "mae": mae,
        "wrmsse_proxy": rnn_wrmsse,
        "training_time_sec": train_time,
        "mean_actual": float(np.mean(y_valid)),
        "mean_prediction": float(np.mean(valid_pred)),
    }
])

display(results)

OUTPUT_DIR = "../results/rnn_model_results"
LOCAL_OUTPUT_DIR = "./rnn_model_results"

# If the Drive path is not available, save locally.
if not os.path.exists(os.path.dirname(OUTPUT_DIR)):
    OUTPUT_DIR = LOCAL_OUTPUT_DIR

os.makedirs(OUTPUT_DIR, exist_ok=True)

results_path = os.path.join(OUTPUT_DIR, "rnn_results_summary.csv")
pred_path = os.path.join(OUTPUT_DIR, "rnn_valid_predictions.csv")

results.to_csv(results_path, index=False)

pred_df = pd.DataFrame({
    "id": valid_ids,
    "date": pd.to_datetime(valid_dates),
    "actual_demand": y_valid,
    "rnn_pred": valid_pred,
})

pred_df.to_csv(pred_path, index=False)

print("Saved results to:", results_path)
print("Saved predictions to:", pred_path)

display(pred_df.head())


,model,seq_len,validation_window_start,validation_window_end,n_series_total,n_features,train_sequences,valid_sequences,max_train_sequences,max_valid_sequences,rmse,mae,wrmsse_proxy,training_time_sec,mean_actual,mean_prediction
0,GRU,28,2016-04-25,2016-05-22,30490,17,500000,853720,500000,None,2.258386,1.149222,0.863086,465.4544,1.44282,1.425817


Saved results to: ./rnn_model_results/rnn_results_summary.csv
Saved predictions to: ./rnn_model_results/rnn_valid_predictions.csv


,id,date,actual_demand,rnn_pred
0,FOODS_1_001_CA_1_evaluation,2016-04-25,2.0,0.586622
1,FOODS_1_001_CA_1_evaluation,2016-04-26,0.0,0.800332
2,FOODS_1_001_CA_1_evaluation,2016-04-27,0.0,1.044106
3,FOODS_1_001_CA_1_evaluation,2016-04-28,0.0,1.056063
4,FOODS_1_001_CA_1_evaluation,2016-04-29,0.0,1.035237


In [15]:
comparison_table = pd.DataFrame([
    {
        "Model": "GRU",
        "RMSE": rmse,
        "MAE": mae,
        "WRMSSE Proxy": rnn_wrmsse,
        "Training Time (sec)": train_time
    }
])

comparison_table = comparison_table.sort_values("RMSE").reset_index(drop=True)

display(comparison_table)

comparison_table.to_csv("model_comparison_with_rnn.csv", index=False)

,Model,RMSE,MAE,WRMSSE Proxy,Training Time (sec)
0,GRU,2.258386,1.149222,0.863086,465.4544


## 14. Report write-up

You can use this in the model comparison section:

> To compare with our main LightGBM model, we trained a lightweight GRU-based RNN model using the same final engineered feature dataset. For consistency with the Random Forest and Ridge Regression model exploration, we applied the same active-row filter, dropped rows with missing required lag, rolling, and price features, used the same final feature columns, and evaluated the model on the same final 28-day validation window.

The main difference is that the GRU model converts the long-format data into sequential samples, using the previous 28 days of features to predict the next-day demand. We report RMSE, MAE, and the same simplified item-store WRMSSE proxy used in the team’s alternative-model exploration notebook.


In [20]:
import os

FEATURE_DATA_PATH = "../results/long_df_with_features.parquet"  # output of notebook 03
SAMPLE_SUBMISSION_PATH = "../data/raw/sample_submission.csv"
CALENDAR_PATH = "../data/raw/calendar.csv"
SELL_PRICES_PATH = "../data/raw/sell_prices.csv"

for p in [FEATURE_DATA_PATH, SAMPLE_SUBMISSION_PATH, CALENDAR_PATH, SELL_PRICES_PATH]:
    print(p, "exists:", os.path.exists(p))


/content/drive/MyDrive/long_df_with_features.parquet exists: True
sample_submission.csv exists: True
calendar.csv exists: True
sell_prices.csv exists: True


In [21]:
cat_cols = ["item_id", "dept_id", "cat_id", "store_id", "state_id"]

num_cols = [c for c in feature_cols if c not in cat_cols]

print("cat_cols:", cat_cols)
print("num_cols:", num_cols)

cat_cols: ['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
num_cols: ['sell_price', 'price_change', 'is_promo', 'dow', 'month', 'lag_7', 'lag_28', 'lag_56', 'rmean_7', 'rmean_28', 'snap_flag', 'is_event']


In [25]:
# =========================
# Generate true Kaggle submission for GRU
# Validation: 2016-04-25 to 2016-05-22
# Evaluation: 2016-05-23 to 2016-06-19
# =========================

import pandas as pd
import numpy as np
import os

# -------------------------
# 1. Load required files
# -------------------------
sample_sub = pd.read_csv(SAMPLE_SUBMISSION_PATH)
calendar = pd.read_csv(CALENDAR_PATH)
sell_prices = pd.read_csv(SELL_PRICES_PATH)

calendar["date"] = pd.to_datetime(calendar["date"])

f_cols = [f"F{i}" for i in range(1, 29)]

print("sample_submission:", sample_sub.shape)
print("calendar:", calendar.shape)
print("sell_prices:", sell_prices.shape)


# -------------------------
# 2. Make sure needed notebook objects exist
# -------------------------

# Define feature groups if not already defined
cat_cols = ["item_id", "dept_id", "cat_id", "store_id", "state_id"]
num_cols = [c for c in feature_cols if c not in cat_cols]

print("cat_cols:", cat_cols)
print("num_cols:", num_cols)

required_objects = [
    "model", "df", "feature_cols", "cat_cols", "num_cols",
    "encoder", "scaler", "SEQ_LEN", "BATCH_SIZE",
    "valid_ids", "valid_dates", "y_valid", "valid_pred"
]

missing_objects = [x for x in required_objects if x not in globals()]
if missing_objects:
    raise NameError(f"Missing objects from previous notebook cells: {missing_objects}")

print("All required notebook objects exist.")


# -------------------------
# 3. Build validation prediction wide table
# -------------------------
pred_df = pd.DataFrame({
    "id": valid_ids,
    "date": pd.to_datetime(valid_dates),
    "actual_demand": y_valid,
    "rnn_pred": valid_pred
})

pred_df = pred_df.sort_values(["id", "date"]).reset_index(drop=True)
pred_df["F_num"] = pred_df.groupby("id").cumcount() + 1
pred_df = pred_df[pred_df["F_num"].between(1, 28)].copy()
pred_df["F"] = "F" + pred_df["F_num"].astype(str)
pred_df["rnn_pred"] = np.clip(pred_df["rnn_pred"], 0, None)

valid_wide = (
    pred_df
    .pivot(index="id", columns="F", values="rnn_pred")
    .reset_index()
)

# Make sure columns are in Kaggle F1-F28 order
valid_wide = valid_wide[["id"] + f_cols]

# Make validation ids match Kaggle sample_submission format
def to_validation_id(x):
    x = str(x)
    if x.endswith("_validation"):
        return x
    elif x.endswith("_evaluation"):
        return x.replace("_evaluation", "_validation")
    else:
        return x + "_validation"

valid_wide["id"] = valid_wide["id"].apply(to_validation_id)

print("Validation wide:", valid_wide.shape)
print("Validation id example:", valid_wide["id"].head().tolist())


# -------------------------
# 4. Prepare id-level information
# -------------------------
id_info = (
    df[["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]]
    .drop_duplicates("id")
    .sort_values("id")
    .reset_index(drop=True)
)

# Keep raw ids without suffix for internal forecasting
id_info["id"] = id_info["id"].astype(str)

def strip_suffix(x):
    x = str(x)
    if x.endswith("_validation"):
        return x.replace("_validation", "")
    elif x.endswith("_evaluation"):
        return x.replace("_evaluation", "")
    else:
        return x

id_info["id_raw"] = id_info["id"].apply(strip_suffix)

# Use original df id format for internal order
id_order = id_info["id"].values
n_ids = len(id_order)

print("Number of series:", n_ids)


# -------------------------
# 5. Build initial sequence array from last 28 available feature rows
# -------------------------
history_raw = df.sort_values(["id", "date"]).reset_index(drop=True).copy()

# Keep ids in stable order
history_raw = history_raw[history_raw["id"].isin(id_order)].copy()
history_raw["id"] = pd.Categorical(history_raw["id"], categories=id_order, ordered=True)
history_raw = history_raw.sort_values(["id", "date"]).reset_index(drop=True)

last_seq_raw = history_raw.groupby("id", sort=False).tail(SEQ_LEN).copy()

check_counts = last_seq_raw.groupby("id").size()
if not (check_counts == SEQ_LEN).all():
    bad_ids = check_counts[check_counts != SEQ_LEN]
    raise ValueError(f"Some ids do not have {SEQ_LEN} rows for initial sequence. Example: {bad_ids.head()}")

last_seq_raw = last_seq_raw.sort_values(["id", "date"]).reset_index(drop=True)

last_seq_encoded = last_seq_raw[feature_cols].copy()

# Encode categorical columns
last_seq_encoded[cat_cols] = encoder.transform(last_seq_encoded[cat_cols])

# Scale numeric columns
last_seq_encoded[num_cols] = scaler.transform(last_seq_encoded[num_cols])

seq_array = (
    last_seq_encoded[feature_cols]
    .astype("float32")
    .values
    .reshape(n_ids, SEQ_LEN, len(feature_cols))
)

print("Initial sequence array:", seq_array.shape)


# -------------------------
# 6. Prepare demand history matrix for recursive lag features
# -------------------------
# Need at least 56 days for lag_56 and rmean_28 based on shift(28).rolling(28)
hist_demand = (
    history_raw[["id", "date", "demand"]]
    .sort_values(["id", "date"])
    .copy()
)

last_90_demand = hist_demand.groupby("id", sort=False).tail(90).copy()

demand_mat = (
    last_90_demand
    .pivot(index="id", columns="date", values="demand")
    .loc[id_order]
    .astype("float32")
    .values
)

if demand_mat.shape[1] < 56:
    raise ValueError("Need at least 56 days of demand history for recursive features.")

print("Demand history matrix:", demand_mat.shape)


# -------------------------
# 7. Prepare future calendar and price features for evaluation period
# -------------------------
eval_start = pd.Timestamp("2016-05-23")
eval_end = pd.Timestamp("2016-06-19")

future_cal = calendar[
    (calendar["date"] >= eval_start) &
    (calendar["date"] <= eval_end)
].copy()

if len(future_cal) != 28:
    raise ValueError(f"Expected 28 future calendar days, got {len(future_cal)}")

future_cal = future_cal[[
    "date", "wm_yr_wk",
    "snap_CA", "snap_TX", "snap_WI",
    "event_name_1"
]].copy()

future_cal["event_name_1"] = future_cal["event_name_1"].fillna("NoEvent")

print("Future calendar period:", future_cal["date"].min(), "to", future_cal["date"].max())


# Build all future id-date rows
future_base = (
    id_info[["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]]
    .assign(key=1)
    .merge(future_cal.assign(key=1), on="key", how="outer")
    .drop(columns="key")
)

future_base = future_base.merge(
    sell_prices,
    on=["store_id", "item_id", "wm_yr_wk"],
    how="left"
)

# Fill missing future sell_price using last known price per id
price_history = history_raw[["id", "date", "sell_price"]].copy()
future_price = future_base[["id", "date", "sell_price"]].copy()

all_price = pd.concat([price_history, future_price], ignore_index=True)
all_price["id"] = pd.Categorical(all_price["id"], categories=id_order, ordered=True)
all_price = all_price.sort_values(["id", "date"]).reset_index(drop=True)

all_price["sell_price"] = (
    all_price
    .groupby("id", sort=False)["sell_price"]
    .ffill()
    .bfill()
)

future_price_filled = all_price[
    (all_price["date"] >= eval_start) &
    (all_price["date"] <= eval_end)
][["id", "date", "sell_price"]].copy()

future_base = future_base.drop(columns=["sell_price"])
future_base = future_base.merge(
    future_price_filled,
    on=["id", "date"],
    how="left"
)

future_base["id"] = pd.Categorical(future_base["id"], categories=id_order, ordered=True)
future_base = future_base.sort_values(["date", "id"]).reset_index(drop=True)

print("Future base:", future_base.shape)


# -------------------------
# 8. Recursive evaluation forecasting
# -------------------------
# Previous price for price_change and is_promo
last_price = (
    history_raw.groupby("id", sort=False)
    .tail(1)
    .set_index("id")
    .loc[id_order, "sell_price"]
    .astype("float32")
    .values
)

eval_preds = []
future_dates = sorted(future_base["date"].unique())

for step, current_date in enumerate(future_dates, start=1):
    day_base = future_base[future_base["date"] == current_date].copy()
    day_base["id"] = pd.Categorical(day_base["id"], categories=id_order, ordered=True)
    day_base = day_base.sort_values("id").reset_index(drop=True)

    if len(day_base) != n_ids:
        raise ValueError(f"{current_date}: expected {n_ids} rows, got {len(day_base)}")

    # Predict demand for this future day using current sequence
    day_pred = model.predict(seq_array, batch_size=BATCH_SIZE, verbose=0).reshape(-1)
    day_pred = np.clip(day_pred, 0, None).astype("float32")
    eval_preds.append(day_pred)

    # Build feature row for this current future date, so it can enter the sequence
    sell_price_arr = day_base["sell_price"].astype("float32").values

    price_change = np.where(
        last_price > 0,
        (sell_price_arr - last_price) / last_price,
        0.0
    ).astype("float32")

    is_promo = (sell_price_arr < last_price).astype("int8")

    # Lag and rolling features for current date, using known + predicted demand history
    lag_7 = demand_mat[:, -7]
    lag_28 = demand_mat[:, -28]
    lag_56 = demand_mat[:, -56]

    # Match shift-first logic:
    # rmean_7 uses demand from t-13 to t-7
    # rmean_28 uses demand from t-55 to t-28
    rmean_7 = demand_mat[:, -13:-6].mean(axis=1)
    rmean_28 = demand_mat[:, -55:-27].mean(axis=1)

    # State-specific SNAP flag
    snap_flag = np.select(
        [
            day_base["state_id"].values == "CA",
            day_base["state_id"].values == "TX",
            day_base["state_id"].values == "WI",
        ],
        [
            day_base["snap_CA"].values,
            day_base["snap_TX"].values,
            day_base["snap_WI"].values,
        ],
        default=0
    ).astype("int8")

    is_event = (day_base["event_name_1"].fillna("NoEvent").values != "NoEvent").astype("int8")

    # Build raw feature dataframe for the current future day
    day_features_raw = day_base[["item_id", "dept_id", "cat_id", "store_id", "state_id"]].copy()

    day_features_raw["sell_price"] = sell_price_arr
    day_features_raw["price_change"] = price_change
    day_features_raw["is_promo"] = is_promo
    day_features_raw["dow"] = pd.to_datetime(current_date).dayofweek
    day_features_raw["month"] = pd.to_datetime(current_date).month
    day_features_raw["lag_7"] = lag_7
    day_features_raw["lag_28"] = lag_28
    day_features_raw["lag_56"] = lag_56
    day_features_raw["rmean_7"] = rmean_7
    day_features_raw["rmean_28"] = rmean_28
    day_features_raw["snap_flag"] = snap_flag
    day_features_raw["is_event"] = is_event

    day_features_raw = day_features_raw[feature_cols].copy()

    # Apply same encoder and scaler from training
    day_features_encoded = day_features_raw.copy()
    day_features_encoded[cat_cols] = encoder.transform(day_features_encoded[cat_cols])
    day_features_encoded[num_cols] = scaler.transform(day_features_encoded[num_cols])

    day_matrix = day_features_encoded[feature_cols].astype("float32").values

    # Update RNN sequence: drop oldest day, append current future feature row
    seq_array = np.concatenate(
        [seq_array[:, 1:, :], day_matrix[:, None, :]],
        axis=1
    )

    # Update recursive demand history
    demand_mat = np.concatenate(
        [demand_mat, day_pred[:, None]],
        axis=1
    )

    # Keep matrix from growing too much
    if demand_mat.shape[1] > 120:
        demand_mat = demand_mat[:, -120:]

    last_price = sell_price_arr

    print(f"Finished recursive forecast day {step}/28: {pd.to_datetime(current_date).date()}")

eval_pred_mat = np.column_stack(eval_preds)

print("Evaluation prediction matrix:", eval_pred_mat.shape)


# -------------------------
# 9. Build evaluation wide table
# -------------------------
eval_wide = pd.DataFrame(eval_pred_mat, columns=f_cols)

def to_evaluation_id(x):
    x = str(x)
    if x.endswith("_evaluation"):
        return x
    elif x.endswith("_validation"):
        return x.replace("_validation", "_evaluation")
    else:
        return x + "_evaluation"

eval_ids = pd.Series(id_order).apply(to_evaluation_id).values
eval_wide.insert(0, "id", eval_ids)

print("Evaluation wide:", eval_wide.shape)
print("Evaluation id example:", eval_wide["id"].head().tolist())


# -------------------------
# 10. Create final Kaggle submission
# -------------------------
submission = sample_sub.copy()
submission[f_cols] = 0.0

# Fill validation rows
valid_map = valid_wide.set_index("id")[f_cols]

validation_ids = submission.loc[
    submission["id"].str.endswith("_validation"),
    "id"
]

matched_validation_ids = validation_ids[validation_ids.isin(valid_map.index)]

submission.loc[
    submission["id"].isin(matched_validation_ids),
    f_cols
] = valid_map.loc[matched_validation_ids].values

# Fill evaluation rows
eval_map = eval_wide.set_index("id")[f_cols]

evaluation_ids = submission.loc[
    submission["id"].str.endswith("_evaluation"),
    "id"
]

matched_evaluation_ids = evaluation_ids[evaluation_ids.isin(eval_map.index)]

submission.loc[
    submission["id"].isin(matched_evaluation_ids),
    f_cols
] = eval_map.loc[matched_evaluation_ids].values


# -------------------------
# 11. Final checks and save
# -------------------------
print("Matched validation rows:", len(matched_validation_ids))
print("Matched evaluation rows:", len(matched_evaluation_ids))
print("Final submission shape:", submission.shape)
print("Expected sample shape:", sample_sub.shape)
print("Missing values:", submission.isna().sum().sum())

validation_rows = submission[submission["id"].str.endswith("_validation")]
evaluation_rows = submission[submission["id"].str.endswith("_evaluation")]

print("Validation nonzero rows:", (validation_rows[f_cols].sum(axis=1) > 0).sum())
print("Evaluation nonzero rows:", (evaluation_rows[f_cols].sum(axis=1) > 0).sum())

print("Validation total forecast:", validation_rows[f_cols].sum().sum())
print("Evaluation total forecast:", evaluation_rows[f_cols].sum().sum())

display(submission.head())
display(submission.tail())

SUBMISSION_PATH = "gru_kaggle_submission.csv"
submission.to_csv(SUBMISSION_PATH, index=False)

print("Saved Kaggle submission:", SUBMISSION_PATH)

sample_submission: (60980, 29)
calendar: (1969, 14)
sell_prices: (6841121, 4)
cat_cols: ['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
num_cols: ['sell_price', 'price_change', 'is_promo', 'dow', 'month', 'lag_7', 'lag_28', 'lag_56', 'rmean_7', 'rmean_28', 'snap_flag', 'is_event']
All required notebook objects exist.
Validation wide: (30490, 29)
Validation id example: ['FOODS_1_001_CA_1_validation', 'FOODS_1_001_CA_2_validation', 'FOODS_1_001_CA_3_validation', 'FOODS_1_001_CA_4_validation', 'FOODS_1_001_TX_1_validation']
Number of series: 30490


/tmp/ipykernel_2091/1628563305.py:133: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  last_seq_raw = history_raw.groupby("id", sort=False).tail(SEQ_LEN).copy()
/tmp/ipykernel_2091/1628563305.py:135: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  check_counts = last_seq_raw.groupby("id").size()


Initial sequence array: (30490, 28, 17)


/tmp/ipykernel_2091/1628563305.py:170: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  last_90_demand = hist_demand.groupby("id", sort=False).tail(90).copy()


Demand history matrix: (30490, 90)
Future calendar period: 2016-05-23 00:00:00 to 2016-06-19 00:00:00


...

/tmp/ipykernel_2091/1628563305.py:263: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  history_raw.groupby("id", sort=False)


Finished recursive forecast day 1/28: 2016-05-23
Finished recursive forecast day 2/28: 2016-05-24
Finished recursive forecast day 3/28: 2016-05-25
Finished recursive forecast day 4/28: 2016-05-26
Finished recursive forecast day 5/28: 2016-05-27
Finished recursive forecast day 6/28: 2016-05-28
Finished recursive forecast day 7/28: 2016-05-29
Finished recursive forecast day 8/28: 2016-05-30
Finished recursive forecast day 9/28: 2016-05-31
Finished recursive forecast day 10/28: 2016-06-01
Finished recursive forecast day 11/28: 2016-06-02
Finished recursive forecast day 12/28: 2016-06-03
Finished recursive forecast day 13/28: 2016-06-04
Finished recursive forecast day 14/28: 2016-06-05
Finished recursive forecast day 15/28: 2016-06-06
Finished recursive forecast day 16/28: 2016-06-07
Finished recursive forecast day 17/28: 2016-06-08
Finished recursive forecast day 18/28: 2016-06-09
Finished recursive forecast day 19/28: 2016-06-10
Finished recursive forecast day 20/28: 2016-06-11
Finished 

,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,...,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
0,HOBBIES_1_001_CA_1_validation,0.778297,0.724663,0.836327,0.863361,1.035945,0.907291,0.872002,1.044433,0.693861,...,0.981786,1.017624,1.015427,0.759659,0.787156,0.931729,0.889946,0.890185,1.052998,1.225668
1,HOBBIES_1_002_CA_1_validation,0.610170,0.450955,0.521370,0.583108,0.587812,0.610923,0.624573,0.692269,0.504208,...,0.564652,0.590308,0.610825,0.624685,0.460521,0.542006,0.505166,0.540584,0.553180,0.635626
2,HOBBIES_1_003_CA_1_validation,1.008471,0.803352,0.985443,0.939554,0.866081,0.893184,0.932860,0.896637,0.653222,...,0.682737,0.846200,0.730887,0.961138,0.640267,0.804761,0.842147,0.840619,0.846463,0.812877
3,HOBBIES_1_004_CA_1_validation,1.697273,1.113817,1.407132,1.055277,1.353928,1.668291,2.698427,2.005028,1.166771,...,1.483055,1.454388,1.612070,1.490654,0.991619,1.210815,1.237754,1.423972,1.270106,1.566965
4,HOBBIES_1_005_CA_1_validation,1.036560,0.710672,0.937496,0.916074,1.042101,0.985070,1.117870,1.682375,1.075898,...,1.092926,1.483424,1.563150,1.609916,1.140951,1.288038,1.348089,1.553361,1.153115,1.164991


,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,...,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
60975,FOODS_3_823_WI_3_evaluation,0.923414,0.609671,0.831934,0.770290,0.845999,0.795307,0.897983,0.767036,0.675234,...,0.841405,0.850273,0.920904,0.963020,0.676880,0.902418,0.842520,0.841598,0.863148,0.898974
60976,FOODS_3_824_WI_3_evaluation,0.643949,0.480553,0.575377,0.571641,0.667172,0.639528,0.750872,0.720281,0.587418,...,0.807478,0.783595,0.899582,0.879714,0.656041,0.839257,0.795233,0.804826,0.823311,0.865709
60977,FOODS_3_825_WI_3_evaluation,1.022270,0.939408,0.796408,0.796894,0.763219,0.868421,0.879981,0.978178,0.670724,...,0.873407,0.870125,0.957192,0.960738,0.735177,0.886729,0.869438,0.832991,0.891428,0.891419
60978,FOODS_3_826_WI_3_evaluation,1.088601,1.011198,1.884255,1.087915,1.501846,1.392455,1.502853,1.342479,0.922687,...,1.081095,0.986834,1.112763,1.069062,0.869298,1.099917,0.983683,1.013909,1.059767,1.049375
60979,FOODS_3_827_WI_3_evaluation,0.899882,1.006778,1.234197,1.069765,1.496271,1.416590,2.029514,1.857013,1.033545,...,1.071490,1.035431,1.190402,1.041196,0.875120,1.041724,0.950186,0.985873,1.007047,1.072920


Saved Kaggle submission: gru_kaggle_submission.csv


In [26]:
f_cols = [f"F{i}" for i in range(1, 29)]

validation_rows = submission[submission["id"].str.endswith("_validation")]
evaluation_rows = submission[submission["id"].str.endswith("_evaluation")]

print("Validation rows:", validation_rows.shape)
print("Evaluation rows:", evaluation_rows.shape)

print("Validation nonzero rows:", (validation_rows[f_cols].sum(axis=1) > 0).sum())
print("Evaluation nonzero rows:", (evaluation_rows[f_cols].sum(axis=1) > 0).sum())

print("Validation total forecast:", validation_rows[f_cols].sum().sum())
print("Evaluation total forecast:", evaluation_rows[f_cols].sum().sum())

Validation rows: (30490, 29)
Evaluation rows: (30490, 29)
Validation nonzero rows: 30490
Evaluation nonzero rows: 30490
Validation total forecast: 1217248.7039006352
Evaluation total forecast: 1224621.2552648634
